In [1]:
from ultralytics import YOLO
import gradio as gr
import pandas as pd
from pathlib import Path
from collections import Counter

# 1. CONFIGURATION

MODEL_PATH = Path("best (1).pt")

CONFIDENCE = 0.339
IMAGE_SIZE = 640

# Check whether the trained model exists
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found at: {MODEL_PATH.resolve()}\n"
        "Place best.pt in the same directory as this script."
    )

# 2. LOAD TRAINED YOLO MODEL

model = YOLO(str(MODEL_PATH))

print("BioSciencia model loaded successfully!")
print("Classes:", model.names)


# 3. IMAGE PREDICTION FUNCTION


def predict_cells(image):

    if image is None:
        return None, None, "Please upload a microscopy image."

    # Run inference only on the uploaded image
    results = model.predict(
        source=image,
        conf=CONFIDENCE,
        imgsz=IMAGE_SIZE,
        save=False,
        verbose=False
    )

    result = results[0]

    # Get annotated image
    annotated_image = result.plot()

    # Count detected classes
    class_counts = Counter()

    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = model.names[class_id]

        class_counts[class_name] += 1

    # Build results table
    all_classes = list(model.names.values())

    table_data = []

    for class_name in all_classes:
        table_data.append({
            "Cell Phase": class_name,
            "Detected Cells": class_counts[class_name]
        })

    dataframe = pd.DataFrame(table_data)

    # Calculate total detected cells
    total_cells = sum(class_counts.values())

    # Build summary
    summary = f"""
## BioSciencia Detection Summary

**Total detected cells: {total_cells}**

- Ana-telophases: {class_counts["ana-telophases"]}
- Interphase: {class_counts["interphase"]}
- Metaphase: {class_counts["metaphase"]}
- Prophase: {class_counts["prophase"]}
"""

    return annotated_image, dataframe, summary


# 4. CREATE INTERACTIVE INTERFACE

with gr.Blocks(title="BioSciencia | Cell Phase Detection") as app:

    gr.Markdown(
        """
        # BioSciencia
        ### Intelligence for Cellular Science

        Upload a microscopy image to detect cells and classify
        them into their respective cell phases.

        **Model:** Custom-trained YOLO  
        **Confidence threshold:** 0.339
        """
    )

    with gr.Row():

        with gr.Column():

            input_image = gr.Image(
                type="numpy",
                label="Upload Microscopy Image"
            )

            detect_button = gr.Button(
                "Detect Cell Phases",
                variant="primary"
            )

        with gr.Column():

            output_image = gr.Image(
                type="numpy",
                label="Annotated Detection Output"
            )

    gr.Markdown("## Detection Results")

    output_table = gr.Dataframe(
        headers=["Cell Phase", "Detected Cells"],
        label="Cell Phase Counts",
        interactive=False
    )

    output_summary = gr.Markdown()

    detect_button.click(
        fn=predict_cells,
        inputs=input_image,
        outputs=[
            output_image,
            output_table,
            output_summary
        ]
    )


# 5. LAUNCH APPLICATION

if __name__ == "__main__":

    app.launch()

BioSciencia model loaded successfully!
Classes: {0: 'ana-telophases', 1: 'interphase', 2: 'metaphase', 3: 'prophase'}
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
